<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista3_3_SistControle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import numpy as np
from numpy.polynomial import Polynomial
import scipy as sp

def polinomioCaracteristico(A):
  A = -A; s = Polynomial([0, 1])
  if(A.shape[0] == 3):
    plus = (s+A[0][0])*(s+A[1][1])*(s+A[2][2]) + (A[0][1]*A[1][2]*A[2][0]) + (A[0][2]*A[1][0]*A[2][1])
    minus = (A[2][0]*(s+A[1][1])*A[0][2]) + (A[2][1]*A[1][2]*(s+A[0][0])) + ((s+A[2][2])*A[1][0]*A[0][1])
    poly = plus-minus; polos = poly.roots()
  elif(A.shape[0] == 2):
    plus = (s+A[0][0])*(s+A[1][1])
    minus = (A[1][0]*A[0][1])
    poly = plus-minus; polos = poly.roots()
  elif(A.shape[0] == 1):
    poly = s+A[0][0]; polos = poly.roots()
  else:
    print("Não é possível calcular o polinômio característico.")
  polos = np.round(polos, 3)
  print("Polinômio característico: " + str(poly))
  print("Polos: " + str(polos))
  for i in range(len(polos)):
    if(polos[i].real > 0):
      print("O sistema é instável.")
      break
  print("O sistema é estável.")

def matrizControlabilidade(A, B):
  n = A.shape[0]
  U = np.zeros((n,n))
  for i in range(n):
    U[:,i:i+1] = (np.linalg.matrix_power(A,i))@B
  posto = np.linalg.matrix_rank(U)
  return U, posto

def matrizObservabilidade(A, C):
  n = A.shape[0]
  V = np.zeros((n,n))
  for i in range(n):
    V[i:i+1, :] = C@(np.linalg.matrix_power(A,i))
  posto = np.linalg.matrix_rank(V)
  return V, posto

def realimentaçãoDeEstado(A, U, s):
  theta = Polynomial.fromroots(s)
  #print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; qc = np.zeros((n,n))
  for i in range(n+1):
    qc += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invU = np.linalg.inv(U)
  linha = np.zeros((1,n)); linha[0][-1] = 1
  return -linha@(invU@qc)

def observadorDeEstado(A, V, s):
  theta = Polynomial.fromroots(s)
  #print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; ql = np.zeros((n,n))
  for i in range(n+1):
    ql += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invV = np.linalg.inv(V)
  coluna = np.zeros((n,1)); coluna[-1][0] = 1
  return ql@invV@coluna

In [22]:
A = np.array([[0, 1],
              [0, 0]])
B = np.array([[0],
              [1]])
C = np.array([[1, 1]])
print("Matrizes do sistema contínuo:");
print("A = " + str(A)); print("B = " + str(C)); print("C = " + str(C))
sys_c = sp.signal.StateSpace(A, B, C)


T = 1; sys_d = sys_c.to_discrete(T)
G = sys_d.A; H = sys_d.B
print("\nDiscretizando o sistema para um período de amostragem de "+str(T)+" segundo(s), obtemos as seguintes matrizes");
print("G = " + str(G)); print("H = " + str(H)); print("C = " + str(C))

Matrizes do sistema contínuo:
A = [[0 1]
 [0 0]]
B = [[1 1]]
C = [[1 1]]

Discretizando o sistema para um período de amostragem de 1 segundo(s), obtemos as seguintes matrizes
G = [[1. 1.]
 [0. 1.]]
H = [[0.5]
 [1. ]]
C = [[1 1]]
